In [3]:
import os, sys
sys.path.insert(0, "C:/Users/ervin/Desktop/world_cup_ML/worldcup_predictor")
from data.classes.load_match import MatchLoader
import pandas as pd

loader = MatchLoader()
df = loader.load()
df = df.dropna(subset=['home_score', 'away_score'])

# ── 1. Basic shape ──────────────────────────────────────────
print(f"Rows:    {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['date'].min()} → {df['date'].max()}")

# ── 2. No nulls in critical columns ────────────────────────
critical_cols = ['date', 'home_team', 'away_team', 
                 'home_score', 'away_score', 'tournament']
print("\nNull counts:")
print(df[critical_cols].isnull().sum())

# ── 3. Date filter worked ───────────────────────────────────
assert df['date'].min() >= pd.Timestamp("2021-01-01"), \
    "❌ Old matches slipping through date filter"
print("\n✅ Date filter correct")

# ── 4. Scores are non-negative integers ─────────────────────
assert (df['home_score'] >= 0).all(), "❌ Negative home scores"
assert (df['away_score'] >= 0).all(), "❌ Negative away scores"
print("✅ Scores are valid")

# ── 5. Weights assigned correctly ───────────────────────────
print("\nWeight distribution:")
print(df.groupby('weight')['tournament'].first().sort_index())

# ── 6. Result column is complete and correct ────────────────
print("\nResult distribution:")
print(df['result'].value_counts())
assert set(df['result'].unique()) == {'home_win', 'away_win', 'draw'}
print("✅ Result column valid")

# ── 7. Spot check a known result ────────────────────────────
# Argentina won 2022 World Cup Final vs France 4-2 on penalties
# The actual 90+ET score was 3-3
final = df[
    (df['home_team'].isin(['Argentina', 'France'])) &
    (df['away_team'].isin(['Argentina', 'France'])) &
    (df['tournament'] == 'FIFA World Cup') &
    (df['date'].dt.year == 2022)
]
print("\n2022 WC Final:")
print(final[['date','home_team','away_team','home_score','away_score']])

[MatchLoader] Loading from cache...
Rows:    5,529
Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'weight', 'goal_diff', 'result']
Date range: 2021-01-12 00:00:00 → 2026-03-31 00:00:00

Null counts:
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
dtype: int64

✅ Date filter correct
✅ Scores are valid

Weight distribution:
weight
0.5                                Friendly
1.0    African Cup of Nations qualification
2.0                     UEFA Nations League
2.5            FIFA World Cup qualification
3.0                           AFC Asian Cup
3.5                               UEFA Euro
4.0                          FIFA World Cup
Name: tournament, dtype: str

Result distribution:
result
home_win    2639
away_win    1624
draw        1266
Name: count, dtype: int64
✅ Result column valid

2022 WC Final:
           date  home_team away_team  home_score  away_score
2063 2022-